# 🔬 Notebook 3: Deep Dive — Cold Starts, Warm Pools, Concurrency & Async

This is the fun one. We **simulate** the execution data plane in pure Python and watch
each technique earn its keep. Every section is a runnable mini-lab with a **bad → best**
progression.

## 🎯 Learning Objectives

1. Feel the difference between a cold start and a warm start, in numbers.
2. Build a **warm pool** with TTL-based garbage collection.
3. Compare concurrency-limiting strategies (none → per-function → per-account).
4. Implement an **async queue with retries + DLQ**.
5. Pick a worker node with **power-of-two choices** to avoid a central bottleneck.
6. Reason about **provisioned concurrency** and **snapshot-based starts** (SnapStart).

## 🛠️ Setup

```bash
cd 06-system-designs/amazon-lambda
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

> 💡 This lab has **no Docker / no cloud dependencies**. Everything is simulated in pure Python
> so you can see the moving parts clearly. Real Lambda uses Firecracker microVMs, S3, DynamoDB,
> and SQS under the hood — we'll point out where those fit as we go.


## 🧊 Part 1: Cold start vs warm start

A cold start is expensive because we do **all** of this from scratch:

1. Download the code `.zip` from S3 — maybe 30 MB.
2. Boot a Firecracker microVM (own kernel, ~125 ms).
3. Start the language runtime (Python/Node/JVM).
4. Run the user's module-level code (imports, constants).
5. Finally call the handler.

A warm start skips steps 1-4 by **re-using a frozen VM** that already did them.

In [1]:
import time, random

# Fake timings in ms — roughly what real Lambda looks like for a small Python function.
DOWNLOAD_MS   = 120     # S3 (cached on worker) -> local disk
VM_BOOT_MS    = 125     # Firecracker microVM boot
RUNTIME_MS    = 150     # Python interpreter + import user module
HANDLER_MS    = 5       # the actual user code

def cold_start_ms():
    return DOWNLOAD_MS + VM_BOOT_MS + RUNTIME_MS + HANDLER_MS

def warm_start_ms():
    return HANDLER_MS    # everything else already paid for

print(f"Cold start: {cold_start_ms()} ms")
print(f"Warm start: {warm_start_ms()} ms")
print(f"Speedup:    {cold_start_ms()/warm_start_ms():.0f}×")

Cold start: 400 ms
Warm start: 5 ms
Speedup:    80×


A ~80× speedup. That's why **every** FaaS trick below is really "get more warm starts".

## 🏊 Part 2: Warm pool (container reuse)

When an invocation finishes, **don't throw the VM away** — freeze it and put it in a pool.
The next invocation for the same function grabs it from the pool in O(1).

After ~10–15 minutes of idleness, the VM is evicted so we don't hog memory forever.

In [2]:
# ❌ Bad: always cold start — throws the VM away after each invocation.
class NoReuse:
    def run(self, fn_name):
        return cold_start_ms()

# ⚠️ Better: keep *all* VMs forever. Fast, but memory grows without bound.
class UnboundedPool:
    def __init__(self):
        self.idle = {}  # fn_name -> list of VMs
    def run(self, fn_name):
        pool = self.idle.setdefault(fn_name, [])
        if pool:
            pool.pop(); return warm_start_ms()
        cost = cold_start_ms()
        pool.append("vm")  # pretend we release it back after executing
        return cost

# ✅ Best: TTL pool with eviction (what real Lambda does).
from collections import defaultdict, deque

class TTLPool:
    def __init__(self, ttl_s=600):   # 10 minutes idle TTL
        self.idle = defaultdict(deque)  # fn -> deque[(vm, idle_since)]
        self.ttl  = ttl_s
        self.cold = self.warm = 0

    def _gc(self, fn, now):
        q = self.idle[fn]
        while q and now - q[0][1] > self.ttl:
            q.popleft()

    def run(self, fn, now=None):
        now = now if now is not None else time.time()
        self._gc(fn, now)
        q = self.idle[fn]
        if q:
            vm, _ = q.popleft()
            self.warm += 1
            latency = warm_start_ms()
        else:
            vm = f"vm-{fn}-{int(now*1000)}"
            self.cold += 1
            latency = cold_start_ms()
        # After running, return the VM to the pool with current timestamp.
        q.append((vm, now))
        return latency


# Simulate 10 000 invocations of 10 functions with realistic bursty traffic.
random.seed(0)
pool = TTLPool(ttl_s=600)
t = 0.0
for _ in range(10_000):
    fn = f"fn-{random.randint(0, 9)}"
    t += random.expovariate(50)   # ~50 invocations per second total
    pool.run(fn, now=t)

total = pool.cold + pool.warm
print(f"Cold: {pool.cold:>5}  Warm: {pool.warm:>5}  Cold-start ratio: {pool.cold/total:.2%}")
avg_latency = (pool.cold*cold_start_ms() + pool.warm*warm_start_ms()) / total
print(f"Avg invocation latency: {avg_latency:.1f} ms (vs {cold_start_ms()} ms with no pool)")

Cold:    10  Warm:  9990  Cold-start ratio: 0.10%
Avg invocation latency: 5.4 ms (vs 400 ms with no pool)


**What to notice**: even a simple pool takes the cold-start ratio from 100 % down to a few percent,
and average latency drops by ~10×. Real systems push cold-start ratios below 1 %.

## 🚦 Part 3: Concurrency limits & throttling

Auto-scaling "to infinity" is a myth. You must cap concurrency per function (and per account)
so that:

- one buggy function can't monopolise a physical server,
- bursts don't exhaust the fleet and break *other* tenants,
- your cost-attack blast radius is bounded.

In [3]:
import threading
from collections import Counter

# ❌ Bad: no limits. A runaway function can eat the whole fleet.
class NoLimit:
    def start(self, account, fn): return object(), "ok"
    def finish(self, token): pass

# ⚠️ Better: per-function semaphore. Excess sync calls get 429.
class PerFunctionLimit:
    def __init__(self, cap=100):
        self.cap = cap; self.sem = {}
    def start(self, account, fn):
        s = self.sem.setdefault((account, fn), threading.Semaphore(self.cap))
        if not s.acquire(blocking=False):
            return None, "429 throttled (function)"
        return s, "ok"
    def finish(self, token): token.release()

# ✅ Best: nested per-account + per-function limits.
# The account cap stops one tenant from monopolising the fleet via many small functions.
class AccountAndFunctionLimit:
    def __init__(self, account_cap=1000, fn_cap=100):
        self.account_cap = account_cap
        self.fn_cap      = fn_cap
        self.account_sem = {}
        self.fn_sem      = {}
    def start(self, account, fn):
        asem = self.account_sem.setdefault(account, threading.Semaphore(self.account_cap))
        fsem = self.fn_sem.setdefault((account, fn), threading.Semaphore(self.fn_cap))
        if not asem.acquire(blocking=False):
            return None, "429 throttled (account)"
        if not fsem.acquire(blocking=False):
            asem.release()
            return None, "429 throttled (function)"
        return (asem, fsem), "ok"
    def finish(self, token):
        asem, fsem = token
        fsem.release(); asem.release()


# Simulate 10 *concurrent in-flight* sync invocations against a per-function cap of 3.
limiter = AccountAndFunctionLimit(account_cap=5, fn_cap=3)
in_flight, results = [], []
for _ in range(10):
    token, status = limiter.start("acct-1", "resize")
    results.append(status)
    if token: in_flight.append(token)

print("Outcomes:", Counter(results))
print(f"In-flight after burst: {len(in_flight)}  (per-function cap was 3)")

# Finish them all; capacity is freed so the next request succeeds again.
for t in in_flight: limiter.finish(t)
_, status = limiter.start("acct-1", "resize")
print("After draining, next request:", status)

Outcomes: Counter({'429 throttled (function)': 7, 'ok': 3})
In-flight after burst: 3  (per-function cap was 3)
After draining, next request: ok


Notice how *some* requests pass and *some* are throttled — not all-or-nothing.
That's the whole point: graceful degradation under pressure, with a clear 429 signal to the client.

## 📬 Part 4: Async invocation — queue + retries + DLQ

Async requests (`X-Invocation-Type: Event`) return **202 Accepted** immediately.
A **poller** reads from an internal queue (SQS/Kafka) and drives the actual work.

Three things we care about:

1. **Retry with exponential backoff** on transient failures (network, out-of-capacity).
2. **Bounded attempts** (usually 3) so we don't loop forever.
3. **Dead-letter queue (DLQ)** for messages that exhaust all attempts, so a human can inspect them.

In [4]:
from collections import deque
import random, itertools

# ❌ Bad: run once, drop on failure. Lossy!
class DropOnFail:
    def __init__(self):
        self.q = deque(); self.lost = 0
    def submit(self, msg): self.q.append(msg)
    def run_once(self, handler):
        while self.q:
            m = self.q.popleft()
            try: handler(m)
            except Exception: self.lost += 1

# ✅ Best: retries with exponential backoff + DLQ.
class AsyncQueue:
    def __init__(self, max_attempts=3, base_delay_s=1.0):
        self.ready = deque()
        self.delayed = []    # (visible_at, msg, attempts)
        self.dlq = []
        self.max = max_attempts
        self.base = base_delay_s

    def submit(self, msg):
        self.ready.append((msg, 0))

    def _reheat(self, now):
        # Move delayed messages whose visibility timer has passed back to ready.
        still = []
        for visible_at, msg, attempts in self.delayed:
            (self.ready.append if visible_at <= now else still.append)((msg, attempts) if visible_at <= now else (visible_at, msg, attempts))
        self.delayed = still

    def drain(self, handler, clock):
        """clock is an iterator of timestamps for determinism in tests."""
        for now in clock:
            self._reheat(now)
            if not self.ready: continue
            msg, attempts = self.ready.popleft()
            try:
                handler(msg)
            except Exception:
                attempts += 1
                if attempts >= self.max:
                    self.dlq.append(msg)
                else:
                    delay = self.base * (2 ** (attempts - 1))   # 1s, 2s, 4s, …
                    self.delayed.append((now + delay, msg, attempts))

# Demo: handler is flaky 70 % of the time.
random.seed(0)
attempts_seen = {}
def flaky_handler(m):
    attempts_seen[m] = attempts_seen.get(m, 0) + 1
    if random.random() < 0.7:
        raise RuntimeError("boom")

q = AsyncQueue(max_attempts=3, base_delay_s=1)
for i in range(20): q.submit(f"msg-{i}")

# Virtual clock: tick every 0.5 s for 30 s.
clock = (i * 0.5 for i in range(60))
q.drain(flaky_handler, clock)

print(f"Completed: {len(attempts_seen) - len(q.dlq)}")
print(f"DLQ:       {len(q.dlq)} -> {q.dlq[:3]}{'…' if len(q.dlq) > 3 else ''}")
print(f"Sample attempt counts: {dict(list(attempts_seen.items())[:5])}")

Completed: 16
DLQ:       4 -> ['msg-5', 'msg-8', 'msg-9']…
Sample attempt counts: {'msg-0': 1, 'msg-1': 1, 'msg-2': 3, 'msg-3': 2, 'msg-4': 2}


The DLQ is the safety net: anything truly broken ends up there for on-call to look at,
instead of silently vanishing. You've now seen all the mechanics real Lambda uses for async events.

## 🎯 Part 5: Scheduling — "power of two choices"

At 600 k QPS, a single central scheduler is a bottleneck. We could round-robin across
worker managers, but that creates hot spots when function/VM sizes vary.

A classic trick: **pick 2 random workers, route to the less-loaded of the two**.
That tiny bit of information-gathering flattens the tail *dramatically* — this is a
well-known result in load-balancing theory.

In [5]:
import random
random.seed(1)

N_WORKERS = 200
workers = [0] * N_WORKERS  # current load per worker

def random_pick():
    return random.randrange(N_WORKERS)

def power_of_two():
    a, b = random.sample(range(N_WORKERS), 2)
    return a if workers[a] <= workers[b] else b

def simulate(pick_fn, n=20_000):
    loads = [0] * N_WORKERS
    for _ in range(n):
        # Rebind workers global so pick_fn sees the current loads.
        global workers; workers = loads
        w = pick_fn()
        loads[w] += 1
    return max(loads), min(loads), sum(loads) / len(loads)

for name, fn in [("random", random_pick), ("power-of-two", power_of_two)]:
    mx, mn, avg = simulate(fn)
    print(f"{name:<13}  max={mx:<4} min={mn:<4} avg={avg:.1f}  (max/avg = {mx/avg:.2f})")

random         max=136  min=77   avg=100.0  (max/avg = 1.36)
power-of-two   max=102  min=96   avg=100.0  (max/avg = 1.02)


Power-of-two-choices brings the worst-case load much closer to the average.
Real Lambda's Placement Service uses a variant of this idea, sharded by function ID.

## 🧊→🚀 Part 6: Provisioned concurrency & SnapStart

Two advanced weapons against cold starts, for completeness:

| Technique | Idea | Trade-off |
|---|---|---|
| **Provisioned concurrency** | Keep N microVMs warm 24/7 for a function; never cold-start until you exceed N. | You pay for idle capacity. |
| **SnapStart / snapshot restore** | Boot the VM once, take a memory snapshot, and *restore* instead of booting. JVM cold starts drop from ~5 s → ~200 ms. | Snapshot correctness (RNG seeds, network sockets) is subtle. |

Below, a tiny simulation of snapshot restore.

In [6]:
# Pretend snapshot restore skips boot + runtime init, but still needs a quick "resume" step.
SNAPSHOT_RESTORE_MS = 50

def snapstart_ms():
    return SNAPSHOT_RESTORE_MS + HANDLER_MS

print(f"Full cold start:    {cold_start_ms()} ms")
print(f"SnapStart restore:  {snapstart_ms()} ms")
print(f"Warm start:         {warm_start_ms()} ms")

Full cold start:    400 ms
SnapStart restore:  55 ms
Warm start:         5 ms


## 🧾 Takeaways

- Cold start is the enemy. **Warm pools** + **code cache on workers** handle 95 % of the problem.
- Concurrency limits per function *and* per account keep one tenant from ruining everyone else's day.
- Async = queue + exponential backoff + DLQ. The DLQ is not optional — it's how you find bugs.
- Scheduling bottlenecks vanish with **power-of-two choices** across sharded worker managers.
- For the last mile: provisioned concurrency (pay for warmth) and SnapStart (skip boot).

## 🔭 What we skipped (and where to look)

- Real **Firecracker** internals: [firecracker-microvm.github.io](https://firecracker-microvm.github.io/).
- Networking isolation (ENI, VPC): real Lambda has a separate control plane just for this.
- Billing at millisecond granularity: out of scope here; conceptually a metering sidecar on each worker.
- Lambda@Edge (run close to the user): same core, different placement strategy.